# V1: Router Agent (Action Autonomy)

This notebook walks through the **CC/CD** (Continuous Calibration / Continuous Deployment) loop for the V1 router:

1. **Build** — instantiate a baseline router that classifies alerts into `infra`, `app`, `security`, `data`.
2. **Test** — evaluate on `data/test_cases_v1.csv`, compute accuracy and per-category breakdown.
3. **Calibrate (CC)** — inspect failures, identify systematic confusions.
4. **Deploy (CD)** — switch to an improved prompt with explicit disambiguation rules, re-evaluate, confirm gains.

In [ ]:
import os, sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')
assert os.environ.get('OPENAI_API_KEY'), 'set OPENAI_API_KEY in ../.env'

import pandas as pd
from src import RouterAgent, evaluate_router

test_cases = pd.read_csv('../data/test_cases_v1.csv')
test_cases.head()

## 1. Baseline router

In [ ]:
baseline_router = RouterAgent(prompt_version='baseline')
baseline_eval = evaluate_router(baseline_router, test_cases)
print(f'Baseline accuracy: {baseline_eval.accuracy:.2%}')
print('Per-category accuracy:')
for cat, acc in baseline_eval.per_category_accuracy.items():
    print(f'  {cat:<10s} {acc:.2%}')

## 2. Calibrate — inspect failures

Look at every wrong prediction and at the confusion matrix to identify systematic errors.

In [ ]:
failures = baseline_eval.predictions[~baseline_eval.predictions.correct]
failures[['id', 'expected', 'predicted', 'alert_text']]

In [ ]:
import json
print('Confusion matrix (rows = expected, cols = predicted):')
print(json.dumps(baseline_eval.confusion, indent=2))

**Common failure patterns to look for**

- Database-server-down (infra) misrouted as `data` because the alert mentions "data".
- Anomalous spend / abuse mis-classified as `app` or `infra` instead of `security`.
- ETL / freshness alerts that mention infrastructure-sounding components mis-routed to `infra`.

The improved prompt encodes explicit rules to resolve these.

## 3. Deploy — improved router

In [ ]:
improved_router = RouterAgent(prompt_version='improved')
improved_eval = evaluate_router(improved_router, test_cases)
print(f'Improved accuracy: {improved_eval.accuracy:.2%}  (baseline was {baseline_eval.accuracy:.2%})')
print('Per-category accuracy:')
for cat, acc in improved_eval.per_category_accuracy.items():
    print(f'  {cat:<10s} {acc:.2%}')

In [ ]:
still_failing = improved_eval.predictions[~improved_eval.predictions.correct]
print(f'{len(still_failing)} cases still failing after improvement')
still_failing[['id', 'expected', 'predicted', 'rationale', 'alert_text']]

## Takeaways

- A short baseline prompt is often enough to get to ~75–85% routing accuracy.
- The remaining gap is almost always concentrated in a small number of confusion patterns.
- Encoding explicit, ordered disambiguation rules in the system prompt is a high-ROI fix — no fine-tuning required.
- The improved router is exported as the input to V2 (the planner).